# Demo Data Pipeline + ML + Chart

This notebook demonstrates an end-to-end mini data pipeline:
1. Create example tabular data
2. Clean and transform data with a preprocessing pipeline
3. Train a machine learning model
4. Evaluate performance
5. Plot a chart


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ----- 1) Build example data -----
n_samples = 500

age = np.random.randint(18, 70, size=n_samples)
monthly_spend = np.random.normal(loc=120, scale=40, size=n_samples).clip(10, None)
visits_per_month = np.random.poisson(lam=5, size=n_samples)
channel = np.random.choice(["web", "mobile", "store"], size=n_samples, p=[0.45, 0.35, 0.20])
region = np.random.choice(["north", "south", "east", "west"], size=n_samples)

# Synthetic binary target: "responded_to_campaign"
score = (
    0.03 * age
    + 0.02 * monthly_spend
    + 0.25 * visits_per_month
    + np.where(channel == "mobile", 1.2, 0)
    + np.where(region == "north", 0.8, 0)
    + np.random.normal(0, 2.0, size=n_samples)
)
responded_to_campaign = (score > np.median(score)).astype(int)

df = pd.DataFrame({
    "age": age,
    "monthly_spend": monthly_spend.round(2),
    "visits_per_month": visits_per_month,
    "channel": channel,
    "region": region,
    "responded_to_campaign": responded_to_campaign
})

# Inject a few missing values to demonstrate imputation
for col in ["monthly_spend", "channel"]:
    missing_idx = np.random.choice(df.index, size=15, replace=False)
    df.loc[missing_idx, col] = np.nan

df.head()

In [ ]:
# ----- 2) Define features and split data -----
target_col = "responded_to_campaign"
X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = ["age", "monthly_spend", "visits_per_month"]
categorical_features = ["channel", "region"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f'Train rows: {len(X_train)}, Test rows: {len(X_test)}')

In [ ]:
# ----- 3) Build data preprocessing + model pipeline -----
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

model = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

clf.fit(X_train, y_train)

In [ ]:
# ----- 4) Evaluate -----
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Accuracy: {acc:.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))

In [ ]:
# ----- 5) Chart output: confusion matrix -----
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, values_format='d', colorbar=False)
ax.set_title('Campaign Response Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Optional extra chart: top feature importances
feature_names = clf.named_steps['preprocessor'].get_feature_names_out()
importances = clf.named_steps['model'].feature_importances_
importance_df = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
ax.set_title('Top 10 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

importance_df